In [ ]:
!pip install datasets # https://pypi.org/project/datasets/
!pip install evaluate
!pip install sentence-transformers
!pip install setfit

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [35]:
from transformers import DataCollatorWithPadding
from transformers import TrainingArguments, Trainer
import numpy as np
import datasets
import evaluate
from setfit import sample_dataset, SetFitModel
from setfit import TrainingArguments as SetFitTrainingArguments
from setfit import Trainer as SetFitTrainer

In [11]:
# Prepare data and splits
tomatoes = load_dataset("rotten_tomatoes")
train_data, test_data = tomatoes["train"], tomatoes["test"]
# prepare data for few shot learning
sampled_train_data = sample_dataset(tomatoes['train'], num_samples=16)

In [17]:
# Data for Supervised Fintuning
print(train_data[0])
print(train_data.num_rows)

{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}
8530


In [18]:
# Data for Sentence Transformer FineTuning
print(sampled_train_data[0])
print(sampled_train_data.num_rows)

{'text': 'escapism in its purest form .', 'label': 1}
32


# Supervised Fine Tuning

In [19]:
# Load model and tokenizer
model_id = "bert-base-cased"
model = AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=2
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [21]:
# Pad to the longest sequence in the batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
def preprocess_function(examples):
   """Tokenize input data"""
   return tokenizer(examples["text"], truncation=True)

# Tokenize train/test data
tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_test = test_data.map(preprocess_function, batched=True)

In [23]:
def compute_metrics(eval_pred):
   """Calculate F1 score"""
   logits, labels = eval_pred
   predictions = np.argmax(logits, axis=-1)

   load_f1 = evaluate.load("f1")
   f1 = load_f1.compute(predictions=predictions, references=labels)["f1"]
   return {"f1": f1}

In [26]:
# # print layers of pre-trained model!!!!
# for name, param in model.named_parameters():
#   print(f"name={name}, weights={param.shape}")

In [24]:
# choosing layers to freeze and train!!!!
for name, param in model.named_parameters():
  # trainable portion
  if name.startswith("classifier"):
    param.requires_grad = True
  # non-trainable portion
  else:
    param.requires_grad = False

In [25]:
# # printing out whether layer is trainable or not
# for name, param in model.named_parameters():
#   print(f"name={name}, trainable={param.requires_grad}")

In [27]:
# Training arguments for parameter tuning
training_args = TrainingArguments(
   "model",
   learning_rate=2e-5,
   per_device_train_batch_size=16,
   per_device_eval_batch_size=16,
   num_train_epochs=1,
   weight_decay=0.01,
   save_strategy="epoch",
   report_to="none"
)
# Trainer which executes the training process
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   processing_class=tokenizer,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)
trainer.train()

Step,Training Loss
500,0.698700


TrainOutput(global_step=534, training_loss=0.6982600322823399, metrics={'train_runtime': 12.2931, 'train_samples_per_second': 693.886, 'train_steps_per_second': 43.439, 'total_flos': 227605451772240.0, 'train_loss': 0.6982600322823399, 'epoch': 1.0})

In [28]:
trainer.evaluate()

{'eval_loss': 0.6821596622467041,
 'eval_f1': 0.6153846153846154,
 'eval_runtime': 1.5693,
 'eval_samples_per_second': 679.292,
 'eval_steps_per_second': 42.695,
 'epoch': 1.0}

In [ ]:
# Load model and tokenizer
model_id = "bert-base-cased"
model_two = AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=2
)
tokenizer_two = AutoTokenizer.from_pretrained(model_id)

In [30]:
# freezing everything before block 11 through indexing!!!
for index, (name, param) in enumerate(model_two.named_parameters()):
    if index < 165:
        param.requires_grad = False
    #print(f"index={index}, name={name}, trainable={param.requires_grad}")


In [31]:
# Trainer which executes the training process
data_collator = DataCollatorWithPadding(tokenizer=tokenizer_two)
trainer = Trainer(
   model=model_two,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   processing_class=tokenizer_two,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)
trainer.train()

Step,Training Loss
500,0.470500


TrainOutput(global_step=534, training_loss=0.46640645609366316, metrics={'train_runtime': 14.0592, 'train_samples_per_second': 606.722, 'train_steps_per_second': 37.982, 'total_flos': 227605451772240.0, 'train_loss': 0.46640645609366316, 'epoch': 1.0})

In [32]:
trainer.evaluate()

{'eval_loss': 0.40933480858802795,
 'eval_f1': 0.8127413127413128,
 'eval_runtime': 1.3495,
 'eval_samples_per_second': 789.936,
 'eval_steps_per_second': 49.649,
 'epoch': 1.0}

# Few-Shot Classification

In [36]:
model_three = SetFitModel.from_pretrained("sentence-transformers/all-mpnet-base-v2")

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


In [37]:
model_three.model_card_data

SetFitModelCardData(language=None, license=None, tags=['setfit', 'sentence-transformers', 'text-classification', 'generated_from_setfit_trainer'], model_name='SetFit with sentence-transformers/all-mpnet-base-v2', model_id=None, dataset_name=None, dataset_id=None, dataset_revision=None, task_name=None, st_id='sentence-transformers/all-mpnet-base-v2', hyperparameters={}, eval_results_dict={}, eval_lines_list=[], metric_lines=[], widget=[], predict_example=None, label_example_list=[], tokenizer_warning=False, train_set_metrics_list=[], train_set_sentences_per_label_list=[], code_carbon_callback=None, num_classes=None, best_model_step=None, metrics=['accuracy'], pipeline_tag='text-classification', library_name='setfit', version={'python': '3.11.12', 'setfit': '1.1.2', 'sentence_transformers': '3.4.1', 'transformers': '4.50.3', 'torch': '2.6.0+cu124', 'datasets': '3.5.0', 'tokenizers': '0.21.1'})

In [46]:
# Define training arguments
args = SetFitTrainingArguments(
    num_epochs=3, # The number of epochs to use for contrastive learning
    num_iterations=20  # The number of text pairs to generate
)
args.eval_strategy = args.evaluation_strategy

# Create trainer
trainer = SetFitTrainer(
    model=model,
    args=args,
    train_dataset=sampled_train_data,
    eval_dataset=test_data,
    metric="f1"
)
# Training loop
trainer.train()

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

***** Running training *****
  Num unique pairs = 1280
  Batch size = 16
  Num epochs = 3


Step,Training Loss,Validation Loss


In [47]:
trainer.evaluate()

***** Running evaluation *****


{'f1': 0.8239771646051379}